# CRANE: Cross-Country Relative Anomaly Network

국가 간 상대가격 네트워크 기반 수입단가 이상충격 조기경보 프레임워크 구현 노트북입니다.

핵심은 행 단위 회귀가 아니라, 같은 `HS code × month` 안에서 공급국가별 수입단가 분포를 만들고 특정 국가가 그 상대가격 네트워크에서 이탈하는지를 측정하는 것입니다.

필수 입력 데이터프레임: `trade_clean`

필수 컬럼: `date`, `hs_code`, `country_code`, `import_value`, `import_weight`. `unit_price`, `log_price`는 없으면 자동 생성합니다.

In [ ]:
# ============================================================
# Cell 1. Imports and settings
# ============================================================
import numpy as np
import pandas as pd

from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import TweedieRegressor, Ridge, Lasso, ElasticNet
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score, roc_auc_score, average_precision_score

import matplotlib.pyplot as plt
from matplotlib.ticker import FuncFormatter

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 160)

COLORS = {"navy":"#243B53", "teal":"#2A9D8F", "coral":"#E76F51", "gold":"#E9C46A", "purple":"#6D597A", "gray":"#6C757D"}

def clean_ax(ax):
    ax.set_facecolor("#FBFBFD")
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.grid(True, alpha=0.25, linestyle="--")
    return ax

print("Ready.")

## 1. 데이터 준비

이미 메모리에 `trade_clean`이 있으면 그대로 사용합니다. CSV를 사용할 경우 주석을 해제하고 경로를 수정하세요.

In [ ]:
# ============================================================
# Cell 2. Prepare trade panel
# ============================================================
# trade_clean = pd.read_csv("data_processed/trade_clean_2020_2024_test.csv")
# trade_clean["date"] = pd.to_datetime(trade_clean["date"])

required_cols = ["date", "hs_code", "country_code", "import_value", "import_weight"]
missing = [c for c in required_cols if c not in trade_clean.columns]
if missing:
    raise ValueError(f"Missing required columns: {missing}")

trade = trade_clean.copy()
trade["date"] = pd.to_datetime(trade["date"])
trade["hs_code"] = trade["hs_code"].astype(str)
trade["country_code"] = trade["country_code"].astype(str)

if "unit_price" not in trade.columns:
    trade["unit_price"] = trade["import_value"] / trade["import_weight"]
if "log_price" not in trade.columns:
    trade["log_price"] = np.log(trade["unit_price"])

trade = trade.replace([np.inf, -np.inf], np.nan)
trade = trade.dropna(subset=["date", "hs_code", "country_code", "import_value", "import_weight", "unit_price", "log_price"])
trade = trade[(trade["import_value"] > 0) & (trade["import_weight"] > 0)].copy()
trade = trade.sort_values(["hs_code", "country_code", "date"]).reset_index(drop=True)

print(trade.shape)
display(trade.head())

## 2. 상대가격 네트워크 feature

동일 `HS × month` 안에서 국가별 로그단가 분포를 만들고 median/MAD 기준 상대가격 이탈 점수 `rel_z`를 계산합니다.

In [ ]:
# ============================================================
# Cell 3. Relative price features
# ============================================================
def mad(x):
    med = np.nanmedian(x)
    return np.nanmedian(np.abs(x - med))

def add_relative_price_features(df, eps=1e-6, min_countries=3):
    out = df.copy()
    grp = out.groupby(["hs_code", "date"])["log_price"]
    out["ht_country_count"] = grp.transform("count")
    out["ht_median_log_price"] = grp.transform("median")
    out["ht_mad_log_price"] = grp.transform(lambda x: mad(x.values)).clip(lower=eps)
    out["rel_z"] = (out["log_price"] - out["ht_median_log_price"]) / out["ht_mad_log_price"]
    out["price_rank_pct"] = out.groupby(["hs_code", "date"])["log_price"].rank(method="average", pct=True)
    out.loc[out["ht_country_count"] < min_countries, ["rel_z", "price_rank_pct"]] = np.nan
    return out

crane = add_relative_price_features(trade)
display(crane[["hs_code", "country_code", "date", "log_price", "ht_country_count", "rel_z", "price_rank_pct"]].head(20))
print("rel_z missing ratio:", crane["rel_z"].isna().mean())

## 3. Leave-one-country-out counterfactual benchmark

각 국가를 평가할 때 자기 자신을 제외한 다른 공급국가들의 median 가격을 기준으로 삼습니다.

In [ ]:
# ============================================================
# Cell 4. Leave-one-country-out benchmark
# ============================================================
def leave_one_out_median_for_group(group):
    vals = group["log_price"].values
    n = len(vals)
    loo = np.full(n, np.nan)
    if n > 2:
        for i in range(n):
            loo[i] = np.median(np.delete(vals, i))
    group["loo_median_log_price"] = loo
    return group

crane = crane.groupby(["hs_code", "date"], group_keys=False).apply(leave_one_out_median_for_group).reset_index(drop=True)
crane["counterfactual_dev"] = crane["log_price"] - crane["loo_median_log_price"]
crane["counterfactual_dev_scaled"] = crane["counterfactual_dev"] / crane["ht_mad_log_price"]

display(crane[["hs_code", "country_code", "date", "log_price", "loo_median_log_price", "counterfactual_dev_scaled"]].head(20))

## 4. Rank instability, persistence, HHI

- `rank_shift`: 전월 대비 상대가격 순위 변화
- `persistence_abs_3m`: 최근 3개월 이상이탈 반복성
- `hhi_hs_month`: HS-월 수입국 집중도

In [ ]:
# ============================================================
# Cell 5. Dynamic network features
# ============================================================
def add_dynamic_network_features(df, k=1.5, lag_window=3):
    out = df.copy().sort_values(["hs_code", "country_code", "date"]).reset_index(drop=True)
    out["lag_price_rank_pct"] = out.groupby(["hs_code", "country_code"])["price_rank_pct"].shift(1)
    out["rank_shift"] = out["price_rank_pct"] - out["lag_price_rank_pct"]
    out["anomaly_abs"] = (out["rel_z"].abs() > k).astype(float)
    out["anomaly_down"] = (out["rel_z"] < -k).astype(float)
    out["anomaly_up"] = (out["rel_z"] > k).astype(float)
    for col in ["anomaly_abs", "anomaly_down", "anomaly_up"]:
        name = col.replace("anomaly_", "")
        out[f"persistence_{name}_{lag_window}m"] = out.groupby(["hs_code", "country_code"])[col].transform(lambda x: x.shift(1).rolling(lag_window, min_periods=1).mean())
    out["total_hs_month_import"] = out.groupby(["hs_code", "date"])["import_value"].transform("sum")
    out["import_share_country"] = out["import_value"] / out["total_hs_month_import"]
    out["hhi_hs_month"] = out.groupby(["hs_code", "date"])["import_share_country"].transform(lambda x: np.sum(x**2))
    out["concentration_risk"] = out["import_share_country"] * out["hhi_hs_month"]
    return out

crane = add_dynamic_network_features(crane, k=1.5, lag_window=3)
show_cols = ["hs_code", "country_code", "date", "rel_z", "price_rank_pct", "rank_shift", "persistence_abs_3m", "import_share_country", "hhi_hs_month", "concentration_risk"]
display(crane[show_cols].head(20))

## 5. Systemic shock score

동일 HS-월 안에서 여러 공급국가가 같은 방향으로 움직이는 정도를 측정합니다.

In [ ]:
# ============================================================
# Cell 6. Systemic sync score
# ============================================================
def add_systemic_sync_score(df):
    out = df.copy().sort_values(["hs_code", "country_code", "date"]).reset_index(drop=True)
    out["price_return"] = out.groupby(["hs_code", "country_code"])["log_price"].diff()
    out["return_sign"] = np.sign(out["price_return"])

    def sync_for_group(g):
        signs = g["return_sign"].dropna()
        if len(signs) == 0:
            g["sync_score"] = np.nan
            g["majority_sign"] = np.nan
            return g
        nonzero = signs[signs != 0]
        majority = 0 if len(nonzero) == 0 else (1 if (nonzero > 0).mean() >= 0.5 else -1)
        g["sync_score"] = (g["return_sign"] == majority).mean()
        g["majority_sign"] = majority
        return g

    return out.groupby(["hs_code", "date"], group_keys=False).apply(sync_for_group).reset_index(drop=True)

crane = add_systemic_sync_score(crane)
display(crane[["hs_code", "country_code", "date", "price_return", "sync_score", "majority_sign"]].head(20))

## 6. 다음 달 네트워크 이상충격 타깃

\[Y^{net}_{hct}=V_{hct}\max(|z_{hc,t+1}|-k,0).\]

In [ ]:
# ============================================================
# Cell 7. Future network anomaly target
# ============================================================
def add_future_anomaly_targets(df, k=1.5):
    out = df.copy().sort_values(["hs_code", "country_code", "date"]).reset_index(drop=True)
    g = out.groupby(["hs_code", "country_code"])
    out["next_rel_z"] = g["rel_z"].shift(-1)
    out["next_date"] = g["date"].shift(-1)
    out["future_abs_anomaly"] = np.maximum(out["next_rel_z"].abs() - k, 0)
    out["future_down_anomaly"] = np.maximum(-out["next_rel_z"] - k, 0)
    out["future_up_anomaly"] = np.maximum(out["next_rel_z"] - k, 0)
    out["future_net_anomaly_amount"] = out["import_value"] * out["future_abs_anomaly"]
    out["future_down_anomaly_amount"] = out["import_value"] * out["future_down_anomaly"]
    out["future_up_anomaly_amount"] = out["import_value"] * out["future_up_anomaly"]
    return out

crane = add_future_anomaly_targets(crane, k=1.5)
print(crane[["future_net_anomaly_amount", "future_down_anomaly_amount", "future_up_anomaly_amount"]].describe())
print("Y=0 ratio:", (crane["future_net_anomaly_amount"] == 0).mean())

## 7. 모델 데이터 구성

In [ ]:
# ============================================================
# Cell 8. Model dataset
# ============================================================
crane["log_import_value"] = np.log(crane["import_value"])
crane["log_import_weight"] = np.log(crane["import_weight"])

crane_features = [
    "rel_z", "counterfactual_dev_scaled", "price_rank_pct", "rank_shift",
    "persistence_abs_3m", "persistence_down_3m", "persistence_up_3m",
    "import_share_country", "hhi_hs_month", "concentration_risk",
    "sync_score", "price_return", "log_price", "log_import_value", "log_import_weight"
]

model_df = crane.replace([np.inf, -np.inf], np.nan).dropna(subset=crane_features + ["future_net_anomaly_amount", "date"])
train_df = model_df[model_df["date"] < "2024-01-01"].copy()
test_df = model_df[model_df["date"] >= "2024-01-01"].copy()

X_train = train_df[crane_features].values
X_test = test_df[crane_features].values
y_train = train_df["future_net_anomaly_amount"].values
y_test = test_df["future_net_anomaly_amount"].values

threshold = np.quantile(y_train, 0.90)
train_df["actual_top10_anomaly"] = (train_df["future_net_anomaly_amount"] >= threshold).astype(int)
test_df["actual_top10_anomaly"] = (test_df["future_net_anomaly_amount"] >= threshold).astype(int)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("train:", train_df.shape, "test:", test_df.shape)
print("target zero ratio train:", (y_train == 0).mean())
print("test base rate:", test_df["actual_top10_anomaly"].mean())
print("features:", crane_features)

## 8. 모델 비교

In [ ]:
# ============================================================
# Cell 9. Model comparison
# ============================================================
def evaluate_predictions(y_true, pred, actual_top10, model_name):
    pred = np.maximum(np.asarray(pred), 0)
    rmse = np.sqrt(mean_squared_error(y_true, pred))
    mae = mean_absolute_error(y_true, pred)
    r2 = r2_score(y_true, pred)
    auc = roc_auc_score(actual_top10, pred) if pd.Series(actual_top10).nunique() > 1 else np.nan
    ap = average_precision_score(actual_top10, pred) if pd.Series(actual_top10).nunique() > 1 else np.nan
    tmp = pd.DataFrame({"y": y_true, "pred": pred, "actual": actual_top10}).sort_values("pred", ascending=False)
    base = tmp["actual"].mean()
    def p_at(k): return tmp.head(min(k, len(tmp)))["actual"].mean()
    def r_at(k):
        total = tmp["actual"].sum()
        return np.nan if total == 0 else tmp.head(min(k, len(tmp)))["actual"].sum() / total
    out = {"model": model_name, "rmse": rmse, "mae": mae, "r2": r2, "auc_top10": auc, "average_precision": ap,
           "precision_at_10": p_at(10), "precision_at_30": p_at(30), "precision_at_50": p_at(50),
           "recall_at_10": r_at(10), "recall_at_30": r_at(30), "recall_at_50": r_at(50), "base_rate": base}
    out["lift_at_10"] = out["precision_at_10"] / base if base > 0 else np.nan
    out["lift_at_30"] = out["precision_at_30"] / base if base > 0 else np.nan
    out["lift_at_50"] = out["precision_at_50"] / base if base > 0 else np.nan
    return out

records = []
actual = test_df["actual_top10_anomaly"].values

records.append(evaluate_predictions(y_test, test_df["import_value"].values, actual, "Baseline: Import Value"))
pred_abs_z_value = test_df["import_value"].values * np.maximum(test_df["rel_z"].abs().values - 1.5, 0)
records.append(evaluate_predictions(y_test, pred_abs_z_value, actual, "Baseline: Current Relative Anomaly x Value"))

X_train_raw = train_df[crane_features].values
X_test_raw = test_df[crane_features].values
y_train_log = np.log1p(y_train)

for name, mdl in {
    "Ridge log-target": Ridge(alpha=1.0),
    "Lasso log-target": Lasso(alpha=0.001, max_iter=20000),
    "ElasticNet log-target": ElasticNet(alpha=0.001, l1_ratio=0.5, max_iter=20000),
}.items():
    pipe = Pipeline([("scaler", StandardScaler()), ("model", mdl)])
    pipe.fit(X_train_raw, y_train_log)
    pred = np.expm1(pipe.predict(X_test_raw))
    records.append(evaluate_predictions(y_test, pred, actual, name))

for name, mdl in {
    "Random Forest": RandomForestRegressor(n_estimators=400, max_depth=8, min_samples_leaf=5, random_state=123, n_jobs=-1),
    "Gradient Boosting": GradientBoostingRegressor(n_estimators=400, learning_rate=0.03, max_depth=3, random_state=123),
}.items():
    mdl.fit(X_train_raw, y_train_log)
    pred = np.expm1(mdl.predict(X_test_raw))
    records.append(evaluate_predictions(y_test, pred, actual, name))

best = None
for power in [1.1, 1.3, 1.5, 1.7, 1.9]:
    for alpha in [0.001, 0.01, 0.1, 1.0, 10.0]:
        tw = TweedieRegressor(power=power, alpha=alpha, link="log", max_iter=10000)
        try:
            tw.fit(X_train_scaled, y_train)
            pred = np.maximum(tw.predict(X_test_scaled), 0)
            rec = evaluate_predictions(y_test, pred, actual, f"Tweedie p={power}, alpha={alpha}")
            rec["power"] = power; rec["alpha"] = alpha
            if best is None or (rec["auc_top10"], -rec["rmse"]) > (best["auc_top10"], -best["rmse"]):
                best = rec; best_tw = tw; best_pred = pred
        except Exception:
            pass

records.append({**best, "model": "Proposed: Tweedie"})
model_comparison = pd.DataFrame(records).sort_values(["auc_top10", "precision_at_30", "rmse"], ascending=[False, False, True]).reset_index(drop=True)
display(model_comparison)

## 9. Risk map과 Top warning list

In [ ]:
# ============================================================
# Cell 10. Risk map and warning table
# ============================================================
test_result = test_df.copy().reset_index(drop=True)
test_result["predicted_net_anomaly_amount"] = best_pred

def pct_rank(s):
    return s.rank(pct=True)

test_result["idiosyncratic_score"] = pct_rank(test_result["rel_z"].abs())
test_result["systemic_score"] = pct_rank(test_result["sync_score"].fillna(test_result["sync_score"].median()))
test_result["amount_score"] = pct_rank(test_result["predicted_net_anomaly_amount"])
test_result["persistence_score"] = pct_rank(test_result["persistence_abs_3m"].fillna(0))
test_result["concentration_score"] = pct_rank(test_result["concentration_risk"].fillna(0))

test_result["composite_risk"] = (
    0.30 * test_result["idiosyncratic_score"]
    + 0.30 * test_result["amount_score"]
    + 0.20 * test_result["persistence_score"]
    + 0.10 * test_result["concentration_score"]
    + 0.10 * (1 - test_result["systemic_score"])
)

top_warning = test_result.sort_values("composite_risk", ascending=False).reset_index(drop=True)
top_warning["rank"] = np.arange(1, len(top_warning) + 1)

cols = ["rank", "hs_code", "country_code", "date", "next_date", "unit_price", "rel_z", "price_rank_pct", "rank_shift", "sync_score", "import_share_country", "hhi_hs_month", "concentration_risk", "future_net_anomaly_amount", "predicted_net_anomaly_amount", "composite_risk"]
display(top_warning[cols].head(30))

fig, ax = plt.subplots(figsize=(8, 6))
sc = ax.scatter(test_result["systemic_score"], test_result["idiosyncratic_score"], s=30 + 120 * test_result["amount_score"], c=test_result["composite_risk"], cmap="viridis", alpha=0.65, edgecolor="none")
ax.axvline(0.5, color=COLORS["gray"], linestyle="--", linewidth=1)
ax.axhline(0.5, color=COLORS["gray"], linestyle="--", linewidth=1)
ax.set_xlabel("Systemic shock score")
ax.set_ylabel("Idiosyncratic anomaly score")
ax.set_title("CRANE Risk Map")
clean_ax(ax)
plt.colorbar(sc, ax=ax, label="Composite risk")
plt.tight_layout()
plt.show()

## 10. Synthetic Injection 검증

같은 HS-월 안에서 특정 국가 가격만 인위적으로 낮춘 뒤, 상대가격 네트워크 이탈 점수가 이를 잘 잡는지 확인합니다.

In [ ]:
# ============================================================
# Cell 11. Synthetic injection validation
# ============================================================
def synthetic_injection_experiment(base_df, inject_frac=0.05, delta=0.30, random_state=123, k=1.5):
    rng = np.random.default_rng(random_state)
    df = base_df.copy().reset_index(drop=True)
    eligible = df[df["import_value"].notna() & df["log_price"].notna()].index.values
    n_inject = max(1, int(len(eligible) * inject_frac))
    inject_idx = rng.choice(eligible, size=n_inject, replace=False)
    df["injected"] = 0
    df.loc[inject_idx, "injected"] = 1
    df.loc[inject_idx, "unit_price"] = df.loc[inject_idx, "unit_price"] * (1 - delta)
    df.loc[inject_idx, "log_price"] = np.log(df.loc[inject_idx, "unit_price"])

    df2 = add_relative_price_features(df)
    df2["injection_score"] = np.maximum(-df2["rel_z"] - k, 0) * df2["import_value"]
    valid = df2[df2["injection_score"].notna()].copy()
    auc = roc_auc_score(valid["injected"], valid["injection_score"]) if valid["injected"].nunique() > 1 else np.nan
    ap = average_precision_score(valid["injected"], valid["injection_score"]) if valid["injected"].nunique() > 1 else np.nan
    sorted_valid = valid.sort_values("injection_score", ascending=False)
    def p_at(k_): return sorted_valid.head(min(k_, len(sorted_valid)))["injected"].mean()
    return {"inject_frac": inject_frac, "delta": delta, "n_injected": int(valid["injected"].sum()), "auc": auc, "average_precision": ap, "precision_at_10": p_at(10), "precision_at_30": p_at(30), "precision_at_50": p_at(50), "base_rate": valid["injected"].mean()}, valid

inj_metrics_list = []
for delta in [0.10, 0.20, 0.30, 0.40]:
    metrics, inj_df = synthetic_injection_experiment(trade, inject_frac=0.03, delta=delta, random_state=123)
    inj_metrics_list.append(metrics)
inj_metrics = pd.DataFrame(inj_metrics_list)
display(inj_metrics)

## 11. 결과 저장

In [ ]:
# ============================================================
# Cell 12. Save outputs
# ============================================================
import os
os.makedirs("output", exist_ok=True)

crane.to_csv("output/crane_features.csv", index=False, encoding="utf-8-sig")
model_comparison.to_csv("output/crane_model_comparison.csv", index=False, encoding="utf-8-sig")
top_warning.to_csv("output/crane_top_warning.csv", index=False, encoding="utf-8-sig")
inj_metrics.to_csv("output/crane_synthetic_injection_metrics.csv", index=False, encoding="utf-8-sig")

print("Saved CRANE outputs to output/ folder.")